# مختبر: أنماط الديكوريترز المتقدمة
في هذا المختبر، ستطبق مفاهيم متقدمة في الديكوريترز (Decorators) بما في ذلك تكديس الديكوريترز، الديكوريترز المبنية بالكلاسات، وتطبيقات واقعية مثل قياس وقت التنفيذ (Timing)، التخزين المؤقت (Caching)، والتحقق من المدخلات (Validation). ستقوم أيضاً ببناء ديكوريتر للتخزين المؤقت (Memoization) يعتمد على وسائط الدالة.

## 1. تكديس الديكوريترز (Stacking Decorators)
عند تطبيق أكثر من ديكوريتر على دالة واحدة، يتم تنفيذها من الداخل إلى الخارج (bottom-up). أي أن الديكوريتر الأقرب إلى تعريف الدالة يُنفذ أولاً. جرب تكديس الديكوريترات التالية ولاحظ تأثير الترتيب.

In [ ]:
# تعريف ديكوريتر uppercase لتحويل الناتج إلى أحرف كبيرة
def uppercase(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

# TODO: أكمل ديكوريتر stars الذي يضيف '***' قبل وبعد الناتج
def stars(func):
    def wrapper(*args, **kwargs):
        # pass  # TODO: أزل هذا السطر وأكمل التنفيذ
        result = func(*args, **kwargs)
        return f"*** {result} ***"
    return wrapper

# TODO: جرب تغيير ترتيب الديكوريترات في السطر التالي (أي @stars فوق @uppercase والعكس) ولاحظ الناتج
@stars
@uppercase
def greet(name):
    return f"Hello, {name}"

print(greet("World"))

## 2. الديكوريترز المبنية بالكلاسات (Class-based Decorators)
يمكن بناء الديكوريترات باستخدام الكلاسات عبر تطبيق الدالة الخاصة `__call__`. هذا مفيد عندما تريد تخزين حالة (state) داخل الديكوريتر. المثال التالي يوضح ديكوريتر يعد عدد مرات استدعاء الدالة.

In [ ]:
class CountCalls:
    """ديكوريتر يقوم بعد مرات استدعاء الدالة"""
    def __init__(self, func):
        self.func = func
        self.count = 0
    
    # TODO: أكمل دالة __call__ لزيادة العداد واستدعاء الدالة
    def __call__(self, *args, **kwargs):
        # pass  # TODO: أزل هذا السطر وأكمل التنفيذ
        self.count += 1
        print(f"استدعاء رقم {self.count} للدالة {self.func.__name__}")
        return self.func(*args, **kwargs)

@CountCalls
def say_hello():
    print("Hello!")

say_hello()
say_hello()
say_hello()

## 3. تطبيق واقعي: قياس وقت التنفيذ (Timing Decorator)
قياس زمن تنفيذ الدوال مفيد لتحسين الأداء. ستقوم ببناء ديكوريتر `timer` الذي يطبع الزمن المستغرق بالثواني.

In [ ]:
import time

# TODO: أكمل ديكوريتر timer لقياس زمن تنفيذ الدالة وطباعته
def timer(func):
    def wrapper(*args, **kwargs):
        # TODO: سجل وقت البداية
        start_time = time.time()
        result = func(*args, **kwargs)
        # TODO: احسب الزمن المستغرق واطبعه
        end_time = time.time()
        elapsed = end_time - start_time
        print(f"الدالة {func.__name__} استغرقت {elapsed:.4f} ثانية")
        return result
    return wrapper

@timer
def slow_function():
    total = 0
    for i in range(1000000):
        total += i
    return total

result = slow_function()
print(f"الناتج: {result}")

## 4. تطبيق واقعي: التخزين المؤقت (Caching Decorator)
التخزين المؤقت يحفظ نتائج الدالة لوسائط تم حسابها مسبقاً لتجنب إعادة الحساب. ستقوم ببناء ديكوريتر `cache` بسيط باستخدام قاموس.

In [ ]:
# TODO: أكمل ديكوريتر cache الذي يخزّن النتائج في قاموس
def cache(func):
    memo = {}
    def wrapper(*args):
        # TODO: تحقق مما إذا كانت النتيجة موجودة في التخزين المؤقت
        if args in memo:
            print("استرجاع من التخزين المؤقت")
            return memo[args]
        # TODO: احسب النتيجة وخزنها
        result = func(*args)
        memo[args] = result
        return result
    return wrapper

@cache
def expensive_computation(n):
    print(f"حساب لـ n={n}...")
    total = sum(range(n))
    return total

print(expensive_computation(1000000))
print(expensive_computation(1000000))  # يجب أن يسترجع من التخزين المؤقت
print(expensive_computation(2000000))

## 5. تطبيق واقعي: التحقق من المدخلات (Input Validation Decorator)
التحقق من أنواع المدخلات يضمن صحة البيانات. ستقوم ببناء ديكوريتر `validate_types` الذي يتحقق من أن وسائط الدالة من الأنواع المحددة.

In [ ]:
from functools import wraps

# TODO: أكمل ديكوريتر validate_types للتحقق من أنواع الوسائط
def validate_types(expected_types):
    """يتوقع expected_types كقائمة من الأنواع لكل وسيط"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # TODO: تحقق من أن عدد الوسائط يطابق عدد الأنواع
            if len(args) != len(expected_types):
                raise ValueError(f"عدد الوسائط {len(args)} لا يطابق عدد الأنواع المتوقعة {len(expected_types)}")
            # TODO: تحقق من نوع كل وسيط
            for i, (arg, t) in enumerate(zip(args, expected_types)):
                if not isinstance(arg, t):
                    raise TypeError(f"الوسيط {i} يجب أن يكون من نوع {t.__name__}، لكن تم إعطاء {type(arg).__name__}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@validate_types([int, str])
def process_data(age, name):
    print(f"معالجة: العمر {age}، الاسم {name}")
    return f"{name} عمره {age}"

# أمثلة صحيحة
print(process_data(25, "Ahmed"))

# أمثلة خاطئة (سترفع أخطاء)
try:
    process_data("25", "Ahmed")  # وسيط أول ليس int
except TypeError as e:
    print(e)

try:
    process_data(25, 30)  # وسيط ثاني ليس str
except TypeError as e:
    print(e)

## 6. تمرين: بناء ديكوريتر Memoization متقدم
في هذا التمرين، ستقوم ببناء ديكوريتر `memoize` يقوم بتخزين نتائج الدالة في خاصية التخزين المؤقت (cache) الخاصة بالدالة المزخرفة. استخدم `functools.wraps` للحفاظ على بيانات الدالة الأصلية.

In [ ]:
from functools import wraps
import time

# TODO: أكمل ديكوريتر memoize باستخدام خاصية التخزين المؤقت في الدالة
def memoize(func):
    @wraps(func)
    def wrapper(*args):
        # إذا لم توجد خاصية cache، قم بإنشائها كقاموس فارغ
        if not hasattr(wrapper, 'cache'):
            wrapper.cache = {}
        # TODO: تحقق من وجود النتيجة في wrapper.cache
        if args in wrapper.cache:
            print("من التخزين المؤقت")
            return wrapper.cache[args]
        # TODO: احسب النتيجة وخزنها في wrapper.cache
        result = func(*args)
        wrapper.cache[args] = result
        return result
    return wrapper

@memoize
def slow_square(n):
    time.sleep(0.5)  # محاكاة عملية بطيئة
    return n * n

# اختبار
print(slow_square(10))
print(slow_square(10))  # من التخزين المؤقت
print(f"حجم التخزين المؤقت: {len(slow_square.cache)}")

## خلاصة
لقد قمت بتطبيق مفاهيم متقدمة للديكوريترز: تكديس الديكوريترز، الديكوريترز المبنية بالكلاسات، وأنماطاً واقعية مثل قياس وقت التنفيذ، التخزين المؤقت، والتحقق من المدخلات. هذه التقنيات تعزز من قدرتك على كتابة كود أكثر أناقة وقابلية للصيانة.